In [2]:
library(dplyr)
library(arrow)
library(ggplot2)
library(xml2)
library(ggsoccer)
library(tidyr)
library(purrr)
library(progressr)
#Progression features
flip_event <- function(event_df){
  gk_x <- event_df |> filter(position == 'TW' & player_team == team_id) |> pull(x)
  needs_flip = gk_x > 0

  if (needs_flip) {
    event_df <- event_df |>
      mutate(
        x = -x,
        ball_x_10frame_forward = -ball_x_10frame_forward,
        ball_x_10frame_forward_p5 = -ball_x_10frame_forward_p5,
        x_p5 = -x_p5,
        x_m5 = -x_m5,
        x_rec = -x_rec
      )
  }
  event_df <- event_df |> mutate(
      x_velo = x_p5 - x_m5,
      dist_from_goal = sqrt((x - goal_coords["x"])**2 + (y - goal_coords["y"])**2)
  )
  return(event_df)
}



In [3]:
#distance, angle (relative to angle to goal), relative velocity between passer and intended receiver (4 features)
get_angle_rel_goal <- function(x1, y1, x2, y2) {
  v1 <- c(goal_coords["x"] - x1, goal_coords["y"] - y1)   # player_1 -> goal
  v2 <- c(goal_coords["x"] - x2, goal_coords["y"] - y2)   # player_2 -> goal
  angle_p1goal_p2goal <- atan2(v1[1]*v2[2] - v1[2]*v2[1], sum(v1*v2))
  abs(angle_p1goal_p2goal) * 180 / pi
}
calc_passer_intended_feats <- function(event_df, opponent_df, teammate_df) {

##distance

  passer_receiver <- event_df |> filter(object_id == player_id | is_intended) 
  pr_distance <- passer_receiver |> select(x, y) |> dist()


## angle
  passer <- passer_receiver |> filter(object_id == player_id)
  receiver <- passer_receiver |> filter(is_intended)
  
  pr_angle <- get_angle_rel_goal(passer$x, passer$y, receiver$x, receiver$y)
  
  #angle_deg

  ## relative velocity

  pr_x_rel_velo = passer$x_velo - receiver$x_velo
  pr_y_rel_velo = passer$y_velo - receiver$y_velo 

  #number of attackers between passer and intended receiver (1 feature)

  closer_to_goal = min(passer$dist_from_goal, receiver$dist_from_goal)
  further_to_goal = max(passer$dist_from_goal, receiver$dist_from_goal)

  num_att_pr <- teammate_df |> filter(closer_to_goal < dist_from_goal, further_to_goal > dist_from_goal) |> nrow()

  #number of defenders between passer and intended receiver (1 feature)
  num_def_pr <- opponent_df |> filter(closer_to_goal < dist_from_goal, further_to_goal > dist_from_goal) |> nrow()
#number of attackers between intended receiver and goal (1 feature)
  num_att_rg <- teammate_df |> filter(receiver$dist_from_goal > dist_from_goal) |> nrow()
#number of defenders between intended receiver and goal (1 feature)
  num_def_rg <- opponent_df |> filter(receiver$dist_from_goal > dist_from_goal) |> nrow()
  print(passer_receiver)
#distance, angle (relative to angle to goal), relative velocity between passer and intended receiver
  return(list(
    pr_dist = pr_distance, 
    pr_angle = pr_angle, 
    pr_x_rel_velo = pr_x_rel_velo, 
    pr_y_rel_velo = pr_y_rel_velo, 
    num_att_pr = num_att_pr,
    num_def_pr = num_def_pr,
    num_att_rg = num_att_rg,
    num_def_rg = num_def_rg) |> as.data.frame())
}


Pressure
For 2 defenders nearest to the intended receiver in front and behind the ball

distance, angle (relative to goal), relative velocity between intended receiver and defender (4 features)
angle formed by passer, intended receiver, and defender (1 feature)

4x5 = 20 features

In [4]:
#distance, angle (relative to goal), relative velocity between intended receiver and defender (4 features)
angle_12_23 <- function(x1, y1, x2, y2, x3, y3) {
  v21 <- c(x1 - x2, y1 - y2)  # vector 2 -> 1
  v23 <- c(x3 - x2, y3 - y2)  # vector 2 -> 3
  
  n1 <- sqrt(sum(v21^2))
  n2 <- sqrt(sum(v23^2))
  if (n1 == 0 || n2 == 0) return(NA_real_)
  
  cos_theta <- sum(v21 * v23) / (n1 * n2)
  cos_theta <- pmax(-1, pmin(1, cos_theta))
  
  acos(cos_theta) * 180 / pi
}
get_f2_b2_players <- function(df, ball, intended){
  df |>
    filter(position != "TW", object_id != player_id, !is_intended) |>
    cross_join(ball, suffix = c("", "_ball")) |>
    cross_join(intended, suffix = c("", "_intended")) |>
    mutate(
      dist_from_intended = sqrt((x - x_intended)^2 + (y - y_intended)^2),
      infront = x > x_ball
    ) |>
    group_by(infront) |>
    arrange(dist_from_intended, .by_group = TRUE) |>
    slice_head(n = 2) |>
    ungroup()
}
calc_f2b2_dists <- function(intended, f2b2_df){
  bind_cols(
  intended |> select(x1 = x, y1 = y),
  f2b2_df |> select(x2 = x, y2 = y, object_id, infront, dist_from_intended)
) |>
  mutate(dist = sqrt((x1 - x2)^2 + (y1 - y2)^2)) |> select(dist, object_id, infront, dist_from_intended)
}

calc_f2b2_angles <- function(intended, f2b2_df){
  bind_cols(
    intended |> select(x1 = x, y1 = y),
    f2b2_df |> select(x2 = x, y2 = y, object_id)
  ) |>
    rowwise() |>
    mutate(angle = get_angle_rel_goal(x1, y1, x2, y2)) |>
    ungroup() |>
    select(angle, object_id)
}

calc_f2b2_rel_velos <- function(intended, t2_def) {
  bind_cols(
  intended |> select(x1_velo = x_velo, y1_velo = y_velo),
  t2_def |> select(x2_velo = x_velo, y2_velo = y_velo, object_id)
) |>
  mutate(
    x_rel_velo = x2_velo - x1_velo,
    y_rel_velo = y2_velo - y1_velo) |> select(x_rel_velo, y_rel_velo, object_id)
}

calc_receiver_defender_feats <- function(event_df, opponent_df, teammate_df, intended, ball) {

  t2_def <- opponent_df |>
    get_f2_b2_players(ball, intended)
  ##distance

dists <- calc_f2b2_dists(intended, t2_def)

## angle
angles <- calc_f2b2_angles(intended, t2_def)

## relative velocity
rel_vels <- calc_f2b2_rel_velos(intended, t2_def)
#angle formed by passer, intended receiver, and defender (1 feature)
pid_angles <- bind_cols(
  intended |> select(i_x = x, i_y = y),
  event_df |> filter(player_id == object_id) |> select(p_x = x, p_y = y),
  t2_def |> select(d_x = x, d_y = y,  object_id)
) |>
  rowwise() |>
  mutate(pid_angle = angle_12_23(p_x, p_y, i_x, i_y, d_x, d_y)) |>
  ungroup() |>
  select(pid_angle, object_id)

opp_defs <- list(dists, angles, rel_vels, pid_angles) 
def_feats <- reduce(opp_defs, merge, by = "object_id")

def_feats  |>
    mutate(side = if_else(infront, "infront", "behind")) |>
    group_by(side) |>
    arrange(dist_from_intended, .by_group = TRUE) |>
    mutate(rnk = row_number()) |>
    filter(rnk <= 2) |>
    ungroup() |>
    complete(side = c("infront", "behind"), rnk = 1:2) |>
    select(side, rnk, dist, angle, x_rel_velo, y_rel_velo, pid_angle) |>
    pivot_wider(
      names_from = c(side, rnk),
      values_from = c(dist, angle, x_rel_velo, y_rel_velo, pid_angle),
      names_glue = "{.value}_{side}_{rnk}_d"
    )
  }


In [5]:
# For 2 teammates nearest to the intended receiver in front and behind the ball

# distance, angle (relative to goal) between intended receiver and teammate (2 features)
# angle formed by passer, intended receiver, and teammate (1 feature)
# distance, angle (relative to angle to goal), relative velocity between teammate and nearest opponent (4 features)

calc_receiver_attacker_feats <- function(teammates, ball, opponent_df, intended) {
  t2_att <- teammates |> get_f2_b2_players(ball, intended)
  # distance, angle (relative to goal) between intended receiver and teammate (2 features)
  dists <- calc_f2b2_dists(intended, t2_att)
  ## angle
  angles <- calc_f2b2_angles(intended, t2_att)

    # angle formed by passer, intended receiver, and teammate (1 feature)

  pit_angles <- bind_cols(
    event_df |> filter(is_intended) |> select(i_x = x, i_y = y),
    event_df |> filter(player_id == object_id) |> select(p_x = x, p_y = y),
    t2_att |> select(d_x = x, d_y = y, object_id)) |>
    rowwise() |>
    mutate(pit_angle = angle_12_23(p_x, p_y, i_x, i_y, d_x, d_y)) |>
    ungroup() |>
    select(pit_angle, object_id)

  # distance, angle (relative to angle to goal), relative velocity between teammate and nearest opponent (4 features)
  ## closest defenders
  ### distance
  closest_opponent <- t2_att |> cross_join(opponent_df |> select(x, y, x_velo, y_velo), suffix = c("", "_def")) |> mutate(
    cls_distance = sqrt((x - x_def)**2 + (y - y_def)**2)
  ) |> group_by(object_id,  .by_group = TRUE) |> arrange(cls_distance) |> slice_head(n = 1) |> ungroup() 

  cls_distance = closest_opponent |> select(cls_distance, object_id)

  ### angle
  cls_angle <- closest_opponent  |>  rowwise() |>
    mutate(cls_angle = get_angle_rel_goal(x, y, x_def, y_def)) |>
    ungroup() |>
    select(cls_angle, object_id)

  cls_rel_velo <- closest_opponent |>
    mutate(
      cls_x_rel_velo = x_velo - x_velo_def,
      cls_y_rel_velo = y_velo - y_velo_def
    ) |> select(cls_x_rel_velo, cls_y_rel_velo, object_id)

  tm_dfs = list(dists, angles, pit_angles, cls_distance, cls_angle, cls_rel_velo)
  tm_feats = reduce(tm_dfs, inner_join, by = "object_id")

  tm_feats <- tm_feats  |>
    mutate(side = if_else(infront, "infront", "behind")) |>
    group_by(side) |>
    arrange(dist_from_intended, .by_group = TRUE) |>
    mutate(rnk = row_number()) |>
    filter(rnk <= 2) |>
    ungroup() |>
    complete(side = c("infront", "behind"), rnk = 1:2) |>
    select(side, rnk, dist, angle, cls_angle, cls_distance, cls_x_rel_velo, cls_y_rel_velo, pit_angle) |>
    pivot_wider(
      names_from = c(side, rnk),
      values_from = c(dist, angle, cls_x_rel_velo, cls_y_rel_velo, pit_angle, cls_angle, cls_distance),
      names_glue = "{.value}_{side}_{rnk}_o"
    )
  
  return(tm_feats)
}

In [6]:
pitch_sportec <- list(
  length = 105,
  width = 68,
  penalty_box_length = 16.5,
  penalty_box_width = 40.32,
  six_yard_box_length = 5.5,
  six_yard_box_width = 18.32,
  penalty_spot_distance = 11,
  goal_width = 7.32,
  origin_x = -52.5,
  origin_y = -34
)

# # A tibble: 6 × 2
#        x      y
#    <dbl>  <dbl>
# 1  -8.01  -0.04
# 2 -15.1   11.7 
# 3 -12.2  -18.1 
# 4  16.3  -29.6 
# 5   6.34  -3.63
# 6  -2.3  -11.0 

event_df|> mutate(
  player_team = ifelse(is_intended, "RECEIVER", player_team),
  player_team = ifelse(object_id == player_id, "PASSER", player_team)      
  )  |> 
  ggplot() + annotate_pitch(dimension = pitch_sportec) + 
  geom_point( aes(x = x, y = y, col = player_team)) + 
  theme_pitch()

: [1m[33mError[39m:[22m
[33m![39m object 'event_df' not found

In [7]:
calc_event_features <- function(event_df) {
  event_df <- flip_event(event_df)
  
  teammate_df <- event_df |> filter(player_team == team_id)
  opponent_df <- event_df |> filter(player_team != team_id & player_team != "BALL")
  ball <- event_df |> filter(player_team == "BALL")
  intended <- event_df |> filter(is_intended)
  
  if (nrow(ball) == 0 || nrow(intended) == 0) {
    return(tibble())
  }
  
  calc_passer_intended_feats(event_df, opponent_df, teammate_df) |>
    bind_cols(
      calc_receiver_defender_feats(event_df, opponent_df, teammate_df, intended, ball),
      calc_receiver_attacker_feats(teammate_df, ball, opponent_df, intended)
    )
}
frames <- read_parquet("/home/lz80/rdf/sp161/shared/soccer-decision-making-r/sportec/passes.parquet") |> filter(is.na(set_piece_type)) |> mutate(
  y_velo = y_p5 - y_m5
)

In [ ]:
#'Location (6 features)
# distance to goal of passer
# angle to goal of passer
# distance to goal of intended receiver
# angle to goal of intended receiver
# x-velocity of intended receiver
# y-velocity of intended receiver

In [17]:
goal_coords = c(x = 52.5, y = 0)
t = frames |> distinct(event_id)
event_df <- frames |> filter(event_id == 18477700000007)
event_df <- flip_event(event_df)
  
teammate_df <- event_df |> filter(player_team == team_id)
opponent_df <- event_df |> filter(player_team != team_id & player_team != "BALL")
ball <- event_df |> filter(player_team == "BALL")
intended <- event_df |> filter(is_intended)
#next - calculate xG in next 10 actions
#calc_event_features(event_df)
#calc_passer_intended_feats(event_df, opponent_df, teammate_df)


event_df |> select(match_id)

# A tibble: 23 × 1
   match_id          
   <chr>             
 1 DFL-MAT-J03YKP.xml
 2 DFL-MAT-J03YKP.xml
 3 DFL-MAT-J03YKP.xml
 4 DFL-MAT-J03YKP.xml
 5 DFL-MAT-J03YKP.xml
 6 DFL-MAT-J03YKP.xml
 7 DFL-MAT-J03YKP.xml
 8 DFL-MAT-J03YKP.xml
 9 DFL-MAT-J03YKP.xml
10 DFL-MAT-J03YKP.xml
# ℹ 13 more rows
# ℹ Use `print(n = ...)` to see more rows

In [27]:
#'Location (6 features)
# distance to goal of passer
# angle to goal of passer
# distance to goal of intended receiver
# angle to goal of intended receiver
# x-velocity of intended receiver
# y-velocity of intended receiver
get_location_feats <- function(event_df, intended) {
  passer <- event_df |> filter(player_id == object_id)
  passer_dist_from_goal <- passer |> select(dist_from_goal) |> pull(1)
  passer_x <- passer |> select(x) |> pull(1)
  passer_y <- passer |> select(y) |> pull(1)

  passer_angle <- get_angle_rel_goal(passer_x, passer_y, 0, 0) 
  #defining angle to goal as angle between vector between player and goal and midpoint and goal

  intended_dist_from_goal <- intended |> select(dist_from_goal) |> pull(1)
  intended_x <- intended |> select(x) |> pull(1)
  intended_y <- intended |> select(y) |> pull(1)

  intended_angle <- get_angle_rel_goal(intended_x, intended_y, 0, 0)
  intended_x_velo <- intended |> select(x_velo) |> pull(1)
  intended_y_velo <- intended |> select(y_velo) |> pull(1)

  tibble(
      p_dist_from_goal = passer_dist_from_goal,
      p_angle = passer_angle,
      i_dist_from_goal = intended_dist_from_goal,
      i_angle <- intended_angle,
      i_x_velo <- intended_x_velo,
      i_y_velo <- intended_y_velo,
    )
  }